# Parcel density map: how the dataset's cover image is made

The cover image of this dataset is a map of Australia drawn from nothing but the parcels themselves: every parcel
has an interior point (`point_lon`, `point_lat`), and each pixel is coloured by how many of those points fall in it.
No basemap, no coastline file: the outline of the country, the cities and the roads between them all come from where
land has been subdivided.

This notebook rebuilds that image step by step, then zooms into a city, where the same method shows individual
streets. It reads only the two point columns, so it needs no geometry library and runs in about a minute.

The pipeline makes the cover with
[`cover_image.py`](https://github.com/nephorion/kaggle-au-cadastre/blob/main/src/cadastre_gnaf/report/cover_image.py),
which does the same thing with numpy and a hand-written PNG encoder. Here matplotlib does the drawing.

In [ ]:
import glob
import math
import os

import matplotlib.pyplot as plt
import numpy as np
import pyarrow.parquet as pq
from matplotlib.colors import LinearSegmentedColormap


def find_data():
    """The dataset folder: wherever Kaggle mounts it, else $CADASTRE_DATA, else cadastre_out here or one up."""
    hits = [p for p in glob.glob("/kaggle/input/**/parquet", recursive=True)
            if glob.glob(f"{p}/*_parcels.parquet")]
    if hits:
        return os.path.dirname(hits[0])
    if os.environ.get("CADASTRE_DATA"):
        return os.environ["CADASTRE_DATA"]
    return next((d for d in ("cadastre_out", "../cadastre_out") if os.path.isdir(d)), "cadastre_out")


DATA = find_data()
FILES = {os.path.basename(f).split("_")[0]: f for f in sorted(glob.glob(f"{DATA}/parquet/*_parcels.parquet"))}
print("dataset:", DATA)
print("states:", ", ".join(FILES))

## 1. Load the interior points

Only `point_lon` and `point_lat` are read, which Parquet can do without touching the other columns. A few parcels
have no point (no usable geometry); they are dropped.

In [ ]:
lons, lats = [], []
for state, path in FILES.items():
    t = pq.read_table(path, columns=["point_lon", "point_lat"])
    lon = t.column("point_lon").to_numpy(zero_copy_only=False)
    lat = t.column("point_lat").to_numpy(zero_copy_only=False)
    ok = ~(np.isnan(lon) | np.isnan(lat))
    lons.append(lon[ok])
    lats.append(lat[ok])
    print(f"{state}: {ok.sum():,} points ({(~ok).sum():,} without a point)")
lon, lat = np.concatenate(lons), np.concatenate(lats)
print(f"total: {len(lon):,} points")

## 2. Count points per pixel

The map covers mainland Australia and Tasmania. Degrees of longitude are shorter than degrees of latitude away from
the equator, so the longitude span is shrunk by `cos(mid latitude)` to keep the country's shape. Then
`np.histogram2d` counts the points in each pixel.

In [ ]:
def density(lon, lat, lon_range, lat_range, height):
    """Point counts on a grid `height` pixels tall over the box, rows running north to south."""
    k = math.cos(math.radians(sum(lat_range) / 2))
    width = round(height * (lon_range[1] - lon_range[0]) * k / (lat_range[1] - lat_range[0]))
    counts, _, _ = np.histogram2d(lat, lon, bins=(height, width), range=(lat_range, lon_range))
    return counts[::-1]                 # histogram rows run south to north; images run top to bottom


LON, LAT = (112.5, 154.0), (-44.0, -9.5)    # the same box as the cover image
counts = density(lon, lat, LON, LAT, height=760)   # fits a 1600 x 800 image with a margin
print(f"grid {counts.shape[1]} x {counts.shape[0]}; {int((counts > 0).sum()):,} pixels have parcels; "
      f"the busiest has {int(counts.max()):,}")

## 3. Why a log scale

Counts are extremely skewed: a pixel in inner Sydney holds thousands of parcels, a pixel in the outback holds one.
On a linear scale only the cities show up. Colouring by `log(1 + count)` instead makes every occupied pixel visible
while keeping the cities brightest.

In [ ]:
occupied = counts[counts > 0]
fig, axes = plt.subplots(1, 2, figsize=(11, 3.2))
axes[0].hist(occupied, bins=100)
axes[0].set_yscale("log")
axes[0].set_title("parcels per occupied pixel")
axes[1].hist(np.log1p(occupied), bins=100)
axes[1].set_title("log(1 + parcels per occupied pixel)")
for ax in axes:
    ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()
print("median parcels per occupied pixel:", int(np.median(occupied)))

## 4. Draw the map

The colour ramp runs dark teal (sparse) to teal, amber and near white (dense), on a dark navy background. Empty
pixels are left transparent so the background shows through.

In [ ]:
BACKGROUND = "#0b1320"
RAMP = LinearSegmentedColormap.from_list("parcels", ["#14465a", "#28aaaa", "#f5be46", "#fffae6"])
RAMP.set_bad(alpha=0)                   # masked (empty) pixels show the background


def draw(counts, size=(16, 8), dpi=100):
    """A borderless figure of `size` inches with the density grid centred on the background."""
    level = np.ma.masked_equal(np.log1p(counts), 0)
    fig = plt.figure(figsize=size, dpi=dpi, facecolor=BACKGROUND)
    ax = fig.add_axes([0, 0, 1, 1], facecolor=BACKGROUND)
    ax.imshow(level, cmap=RAMP, interpolation="nearest")
    h, w = counts.shape
    ax.set_xlim(-(size[0] * dpi - w) / 2, w + (size[0] * dpi - w) / 2)   # pad to the figure's aspect ratio
    ax.set_ylim(h + (size[1] * dpi - h) / 2, -(size[1] * dpi - h) / 2)
    ax.axis("off")
    return fig


fig = draw(counts)
out = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."
fig.savefig(f"{out}/parcel_density.png", facecolor=BACKGROUND)
plt.show()
print(f"saved {out}/parcel_density.png (1600 x 800)")

## 5. Zoom into a city

The same three steps on a small box, with points filtered to it first. At city scale each pixel is a few dozen
metres, so streets, parks (dark, no parcels) and dense apartment blocks (bright, many strata lots) appear. Change
`CITY` to any box you like.

In [ ]:
CITIES = {
    "Sydney": ((150.9, 151.35), (-34.05, -33.7)),
    "Melbourne": ((144.8, 145.2), (-38.0, -37.7)),
    "Brisbane": ((152.9, 153.2), (-27.6, -27.35)),
    "Perth": ((115.7, 116.0), (-32.1, -31.8)),
    "Hobart": ((147.2, 147.45), (-42.95, -42.8)),
    "Canberra": ((149.0, 149.25), (-35.45, -35.15)),
}
CITY = "Sydney"

(lo0, lo1), (la0, la1) = CITIES[CITY]
inside = (lon >= lo0) & (lon <= lo1) & (lat >= la0) & (lat <= la1)
city = density(lon[inside], lat[inside], (lo0, lo1), (la0, la1), height=900)
print(f"{CITY}: {inside.sum():,} parcels")
draw(city, size=(city.shape[1] / 90, city.shape[0] / 90), dpi=90)
plt.show()

## Ideas to try

- Colour by another column instead of the count: the share of strata lots per pixel, or the mean `area_m2`.
- Draw only parcels with no G-NAF address (`gnaf_address_count == 0`) to see where linkage is weakest.
- Use `height=4000` (and a bigger `size`) for a poster-sized map; the counting step stays fast because it never reads geometry.